<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">وزن‌های مشترک، پردازش جداگانهٔ هر موقعیت</h1>
<p style="text-align:right">درس 43 از 92 · بعد از ارتباط، روی ویژگی‌ها چه محاسبه‌ای کنیم؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">37-ffn</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right"><bdi dir="ltr">FFN</bdi> واقعی را از وزن‌هایش بازسازی کنید و استقلال موقعیت‌ها را بسنجید.</p><p style="text-align:right"><span class="phrase-lead" style="white-space:nowrap">پیش‌نیاز: دو</span> <bdi dir="ltr">Linear</bdi> با <bdi dir="ltr">Activation</bdi> میان آن‌ها و محور آخر ویژگی را بشناسید.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۶۰–۱۰۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر دو موقعیت ورودی یکسان باشند، <bdi dir="ltr">FFN</bdi> با وزن مشترک چه خروجی‌هایی می‌دهد؟ تغییر موقعیت سوم به آن دو راهی دارد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
from torch.nn import functional as F
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import FeedForward
ffn = FeedForward(ModelConfig(12,8,4,1,1,0.)).eval()
x = torch.randn(2,3,4)
x[:,1] = x[:,0]
print('matching positions:',torch.equal(x[:,0],x[:,1]))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">manual_ffn(ffn, x)</code> را با وزن و <bdi dir="ltr">Bias Layer</bdi>‌های ۰ و ۲ در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ffn.layers</code> بنویسید. بین دو تبدیل <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">F.gelu</code> به کار ببرید؛ خود <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ffn(x)</code> یا <bdi dir="ltr">Sequential</bdi> کامل را صدا نزنید. <bdi dir="ltr">Dropout</bdi> این آزمایش صفر است.</p>
</div>

In [ ]:
def manual_ffn(ffn, x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = manual_ffn(ffn,x)
    if result is None: return False
    torch.testing.assert_close(result,ffn(x))
    torch.testing.assert_close(result[:,0],result[:,1])
    changed = x.clone(); changed[:,2] += 20
    torch.testing.assert_close(manual_ffn(ffn,changed)[:,:2],result[:,:2])
    other = torch.randn(1,5,4)
    torch.testing.assert_close(manual_ffn(ffn,other),ffn(other))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط موقعیت آخر را به اندازهٔ ۱۰ افزایش دهید. اختلاف خروجی هر موقعیت را جدا چاپ کنید؛ صفرهای سایر موقعیت‌ها مهم‌تر از مقدار عددی اختلاف آخرند.</p>
</div>

In [ ]:
changed = x.clone(); changed[:,-1] += 10
with torch.no_grad():
    print('per-position change:',(ffn(changed)-ffn(x)).abs().amax(-1))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب قبل از <bdi dir="ltr">FFN</bdi> میانگین زمان را می‌گیرد و به همهٔ موقعیت‌ها می‌دهد. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">local_ffn(ffn,x)</code> را اصلاح کنید؛ در این بخش می‌توانید خود <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">ffn</code> را فراخوانی کنید.</p>
</div>

In [ ]:
with torch.no_grad():
    wrong = ffn(x.mean(1,keepdim=True)).expand_as(x)
print('wrong: all positions identical:',torch.equal(wrong[:,0],wrong[:,2]))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def local_ffn(ffn, x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = local_ffn(ffn,x)
    if result is None: return False
    torch.testing.assert_close(result,ffn(x))
    changed = x.clone(); changed[:,2] -= 100
    torch.testing.assert_close(local_ffn(ffn,changed)[:,:2],result[:,:2])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right">این همان <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">FeedForward</code> در <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">mini_gpt/transformer.py</code> است. <bdi dir="ltr">FFN</bdi> اطلاعات همسایه را مستقیماً نمی‌خواند؛ اطلاعات گذشته پیش‌تر از راه <bdi dir="ltr">Attention</bdi> وارد نمایش آن موقعیت شده است.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">اگر <bdi dir="ltr">FFN</bdi> را تنها نگه داریم، تغییر گذشته از کدام مسیر می‌تواند به <bdi dir="ltr">Token</bdi> فعلی برسد؟ چرا وزن مشترک با ورودی مشترک یکی نیست؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-02/37-ffn.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/37-ffn.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>